# Module 06: LoRA/PEFT Basics

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, you'll see what happens when we apply LoRA to
ModernBERT-base. No GPU needed, no training — just parameter counts
that show why this technique is so popular.

## Setup

We need `transformers` for the model and `peft` for LoRA.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

## Step 1: Load ModernBERT-base for classification

We load the same model we use all semester, configured for our
complaint classification task (113 categories after label cleanup in Week 1). This is the "full" model
with all parameters trainable by default.

In [ ]:
model_name = "answerdotai/ModernBERT-base"
num_labels = 111

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
)

## Step 2: Count parameters BEFORE LoRA

Right now, every parameter in the model is trainable.
Let's count them.

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.1f}%")
print()
print("Right now, ALL parameters are trainable.")
print("Full fine-tuning would update every single one of these.")

## Step 3: Look at the model structure

Before applying LoRA, let's see what layers exist so we know
what we're targeting. We'll look for the attention-related layers.

In [ ]:
print("Layers containing 'attn' or 'attention':\n")
for name, module in model.named_modules():
    if "attn" in name.lower() or "attention" in name.lower():
        if hasattr(module, "weight"):
            print(f"  {name}: {module.__class__.__name__} — weight shape {tuple(module.weight.shape)}")

## Step 4: Apply LoRA

Now we apply LoRA. We:
1. Freeze all the original weights
2. Add small trainable matrices alongside certain layers

We use `target_modules="all-linear"` which tells PEFT to add
LoRA adapters to all linear layers. This is a simple, reliable
default. The rank (`r=8`) controls how small the adapter matrices
are — lower rank = fewer parameters.

In [ ]:
lora_config = LoraConfig(
    r=8,                              # rank of the adapter matrices
    lora_alpha=16,                    # scaling factor
    lora_dropout=0.1,                 # dropout on adapter layers
    target_modules="all-linear",      # apply to all linear layers
    bias="none",                      # don't train bias terms
    task_type="SEQ_CLS",              # sequence classification
)

model_with_lora = get_peft_model(model, lora_config)

## Step 5: Count parameters AFTER LoRA

Now let's see what changed. The base model weights are frozen,
and only the LoRA adapter weights are trainable.

In [ ]:
model_with_lora.print_trainable_parameters()

In [ ]:
total_after = sum(p.numel() for p in model_with_lora.parameters())
trainable_after = sum(p.numel() for p in model_with_lora.parameters() if p.requires_grad)
frozen_after = total_after - trainable_after

print(f"\n{'='*55}")
print(f"  BEFORE LoRA:")
print(f"    Total parameters:     {total_params:>12,}")
print(f"    Trainable parameters: {trainable_params:>12,} (100%)")
print(f"")
print(f"  AFTER LoRA:")
print(f"    Total parameters:     {total_after:>12,}")
print(f"    Frozen parameters:    {frozen_after:>12,}")
print(f"    Trainable parameters: {trainable_after:>12,} ({100 * trainable_after / total_after:.2f}%)")
print(f"{'='*55}")
print(f"\n  We went from training {total_params:,} parameters")
print(f"  to training {trainable_after:,} parameters ({100 * trainable_after / total_after:.2f}%)")
print(f"\n  That's a {total_params / trainable_after:.0f}x reduction.")

## Step 6: See which layers are trainable

Let's look at the model to see which specific layers are frozen
(the original weights) vs. trainable (the LoRA adapters).

In [ ]:
print("Trainable layers (LoRA adapters):\n")
for name, param in model_with_lora.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {tuple(param.shape)} — {param.numel():,} params")

In [ ]:
print("\nFrozen layers (first 10):\n")
frozen_count = 0
for name, param in model_with_lora.named_parameters():
    if not param.requires_grad:
        print(f"  {name}: {tuple(param.shape)} — {param.numel():,} params")
        frozen_count += 1
        if frozen_count >= 10:
            print(f"  ... and many more")
            break

## What just happened?

1. We loaded ModernBERT-base with all ~149M parameters trainable
2. We applied LoRA, which **froze** the original weights and **added**
   small adapter matrices
3. Only the adapter matrices are trainable — a tiny fraction of the total
4. During inference, the adapter output gets added to the frozen layer output

The model architecture is the same. The pretrained knowledge is preserved.
We just dramatically reduced what we need to train.

## Check yourself

Before moving on, make sure you can answer these:

1. What does PEFT stand for, and what problem does it solve?
2. What does LoRA do to the pretrained weights? (Hint: it does NOT change them.)
3. Roughly what fraction of parameters are trainable with LoRA vs. full fine-tuning?
4. What is the trade-off? What might you lose by using LoRA instead of full fine-tuning?
5. Why would smaller checkpoints matter in practice?

**You'll implement LoRA fine-tuning for real in Week 3.**